# Differential gene expression analysis

To perform differential gene expression analysis we have several alternatives and modes:
- Single-cell level
- Pseudobulk level

The `dotools_py` package includes two functions to automatically test for DEA between two conditions
for all the celltypes we have defined in our object, as well a consensus function to run both approaches.

In [1]:
# Set up
import anndata as ad
import dotools_py as do

adata = ad.read_h5ad('/Users/david/Downloads/Data10x/adata.h5ad')
adata

2025-07-08 16:31:00,418 - Jupyter enviroment detected. Using "inline" backend


AnnData object with n_obs × n_vars = 2801 × 18517
    obs: 'batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts', 'doublet_class', 'doublet_score', 'leiden', 'cell_type', 'autoAnnot', 'celltypist_conf_score', 'annotation', 'annotation_recluster'
    var: 'mean', 'std', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'annotation_colors', 'annotation_recluster_colors', 'batch_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_CCA', 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'logcounts', 'scaled'
    obsp: 'connectivities', 'distances'

## DEA at the single-cell level

Among the test we can use we have: wilcoxon, t-test, logistic regression, t-test with overestimation of the variance and the MAST test.
The MAST test can be run using the `do.tl.run_mast`, the other test can be run with `do.tl.rank_genes_groups`. Alternatively, we can use `do.tl.rank_genes_condition`, to use any of the test and automatically test for all the cell-types

To reduce the computation time, we are going to only use the NK cells

In [2]:
nk = adata[adata.obs.annotation == 'NK'].copy()

df = do.tl.rank_genes_condition(nk,
                                groupby='condition',
                                subset_by='annotation',
                                reference='healthy',
                                groups=['disease'],
                                method='mast',
                                get_results=True
                                )

2025-07-08 15:57:07,374 - Running DGEs for NK.
2025-07-08 15:57:07,376 - Running MAST test in R.
2025-07-08 15:57:07,444 - Running test for disease


Reading AnnData in R
`fData` has no primerid.  I'll make something up.
`cData` has no wellKey.  I'll make something up.
Assuming data assay in position 1, with name et is log-transformed.
Running MAST Test

Done!
Combining coefficients and standard errors
Calculating log-fold changes
Calculating likelihood ratio tests
Refitting on reduced model...

Done!
Saving DGE Table


In [3]:
df.head(10)

,GeneName,pvals,log2fc,padj,pts_ref,pts_group,groups,annotation
0,A1BG,0.001228,2.602636,0.022698,0.041995,0.195652,disease,NK
1,A1BG-AS1,0.779995,1.540265,1.000000,0.002625,0.000000,disease,NK
2,A2M,0.637401,0.069724,1.000000,0.005249,0.000000,disease,NK
3,A2M-AS1,0.526081,-0.222878,1.000000,0.175853,0.130435,disease,NK
4,A4GALT,1.000000,3.050085,1.000000,0.000000,0.000000,disease,NK
5,AAAS,0.688836,-0.761130,1.000000,0.047244,0.021739,disease,NK
6,AACS,0.381421,-1.352886,0.973504,0.013123,0.000000,disease,NK
7,AAED1,0.044028,0.794737,0.313925,0.091864,0.065217,disease,NK
8,AAGAB,0.524845,0.875611,1.000000,0.065617,0.108696,disease,NK
9,AAK1,0.223586,-0.585253,0.758544,0.464567,0.347826,disease,NK


## DEA at the pseudobulk level

To perform differential gene expression using a pseudobulk approach we can use `do.tl.rank_genes_pseudobulk`, which test between two conditions for each cell-type. We can use `DESEq2` or `edgeR`. In this case we need to generate pseudo-replicates since we only have one sample per condition.

In [6]:
df = do.tl.rank_genes_pseudobulk(adata,
                                 ctrl_cond='healthy',
                                 disease_cond='disease',
                                 cluster_key='annotation',
                                 batch_key='batch',
                                 condition_key='condition',
                                 design='~condition',
                                 min_cells=30,
                                 min_counts=10,
                                 method='deseq2',
                                 pseudobulk_approach='sum',
                                 technical_replicates=2,
                                 get_results=True
                                 )

2025-07-08 16:00:14,527 - Generating Pseudo-bulk data


Pseudo-bulked clusters:   0%|          | 0/5 [00:00<?, ?it/s]



2025-07-08 16:00:14,569 - The samples ['batch2'] have < 30 in cluster Monocytes. Skipping cluster


2025-07-08 16:00:59,523 - Removed 7269 genes for having less than 10 total counts


Pseudo-bulked clusters:  40%|████      | 2/5 [00:44<01:07, 22.50s/it]



2025-07-08 16:01:34,585 - Removed 10335 genes for having less than 10 total counts


...

Fitting dispersions...
... done in 0.39 seconds.

Fitting dispersion trend curve...
... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.63 seconds.

Fitting LFCs...
... done in 0.33 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.22 seconds.

Fitting size factors...
... done in 0.00 seconds.



Log2 fold change & Wald test p-value: condition disease vs healthy
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
A1BG       9.771698        2.687972  0.650315  4.133339  0.000036  0.000315
A1BG-AS1   0.000000             NaN       NaN       NaN       NaN       NaN
A2M-AS1   11.291117       -0.313524  0.640911 -0.489185  0.624711  0.743985
A4GALT     0.000000             NaN       NaN       NaN       NaN       NaN
AAAS       2.260600       -1.056359  1.511954 -0.698671  0.484757       NaN
...             ...             ...       ...       ...       ...       ...
ZXDB       0.000000             NaN       NaN       NaN       NaN       NaN
ZXDC       1.423852       -2.420790  2.482675 -0.975073  0.329524       NaN
ZYG11B     0.000000             NaN       NaN       NaN       NaN       NaN
ZYX       12.240107       -1.696707  0.750275 -2.261446  0.023732  0.076845
ZZEF1      3.512702       -3.724314  2.418610 -1.539857  0.123595       NaN

[11575 rows x 6 colu

Fitting dispersions...
... done in 0.29 seconds.

Fitting dispersion trend curve...
... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.53 seconds.

Fitting LFCs...
... done in 0.30 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.

Running Wald tests...


Log2 fold change & Wald test p-value: condition disease vs healthy
          baseMean  log2FoldChange     lfcSE      stat    pvalue  padj
A1BG      6.758366       -0.052840  0.966360 -0.054680  0.956393   NaN
A1BG-AS1  0.000000             NaN       NaN       NaN       NaN   NaN
A2M-AS1   0.000000             NaN       NaN       NaN       NaN   NaN
A4GALT    2.763818        4.156685  2.713923  1.531615  0.125618   NaN
AAAS      0.000000             NaN       NaN       NaN       NaN   NaN
...            ...             ...       ...       ...       ...   ...
ZXDB      0.000000             NaN       NaN       NaN       NaN   NaN
ZXDC      2.889922       -0.343833  1.439375 -0.238876  0.811202   NaN
ZYG11B    2.803470        1.089600  1.624036  0.670921  0.502271   NaN
ZYX       7.131340       -0.540559  0.949183 -0.569499  0.569018   NaN
ZZEF1     2.645399       -0.474618  1.593758 -0.297798  0.765857   NaN

[11575 rows x 6 columns]


... done in 0.30 seconds.



In [7]:
df.head(10)

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,group
A1BG,41.023510,-0.326821,0.391098,-0.835650,4.033520e-01,0.607094,T_cells
A1BG-AS1,4.028301,-0.892561,1.300960,-0.686079,4.926633e-01,0.682063,T_cells
A2M-AS1,4.782241,-1.267590,1.214172,-1.043996,2.964875e-01,0.504441,T_cells
A4GALT,0.000000,NaN,NaN,NaN,NaN,1.000000,T_cells
AAAS,15.427053,-1.645663,0.720415,-2.284327,2.235232e-02,0.080126,T_cells
AACS,7.095004,0.556354,0.903691,0.615646,5.381279e-01,0.718470,T_cells
AAED1,48.668054,1.877274,0.351991,5.333298,9.644479e-08,0.000001,T_cells
AAGAB,17.122063,0.114797,0.587964,0.195244,8.452017e-01,0.919049,T_cells
AAK1,265.970082,-0.270280,0.151519,-1.783800,7.445617e-02,0.197681,T_cells
AAMDC,9.601830,-0.649625,0.798883,-0.813167,4.161224e-01,0.617768,T_cells


## DEA consensus

Additionally, the `do.tl.rank_genes_consensus` allow to perform both single-cell and pseudo-bulk DEA and generate a dataframe that summarises everything.

In [2]:
df = do.tl.rank_genes_consensus(adata,
                                ctrl_cond='healthy',
                                disease_cond='disease',
                                cluster_key='annotation',
                                batch_key='batch',
                                condition_key='condition',
                                min_cells=30,
                                min_counts=10,
                                pseudobulk_approach='sum',
                                technical_replicates=2,
                                get_results=True,
                                test_pseudobulk='edger',
                                test='wilcoxon'
                                )

2025-07-08 16:31:00,769 - Running wilcoxon
2025-07-08 16:31:00,837 - Running DGEs for B_cells.
2025-07-08 16:31:00,840 - Running wilcoxon test.
ranking genes


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


    finished (0:00:00)
2025-07-08 16:31:01,100 - Running DGEs for Monocytes.
2025-07-08 16:31:01,103 - Running wilcoxon test.
ranking genes
    finished (0:00:00)
2025-07-08 16:31:01,332 - Running DGEs for NK.
2025-07-08 16:31:01,336 - Running wilcoxon test.
ranking genes
    finished (0:00:00)
2025-07-08 16:31:01,738 - Running DGEs for T_cells.
2025-07-08 16:31:01,741 - Running wilcoxon test.
ranking genes
    finished (0:00:01)
2025-07-08 16:31:03,606 - Running DGEs for pDC.
2025-07-08 16:31:03,614 - Running wilcoxon test.
ranking genes
    finished (0:00:00)
2025-07-08 16:31:03,672 - Running edger
2025-07-08 16:31:03,673 - Generating Pseudo-bulk data


Pseudo-bulked clusters:   0%|          | 0/5 [00:00<?, ?it/s]



2025-07-08 16:31:03,688 - The samples ['batch2'] have < 30 in cluster Monocytes. Skipping cluster
2025-07-08 16:31:48,498 - Removed 7269 genes for having less than 10 total counts


...

Generating DGE Table to pass to Python
Reading AnnData in R
Running edgeR Test


2025-07-08 16:33:11,931 - Running DEA for B_cells


Generating DGE Table to pass to Python
Reading AnnData in R
Running edgeR Test


2025-07-08 16:33:19,043 - Generating consensus DataFrame


Generating DGE Table to pass to Python


In [3]:
df.head(10)

,GeneName,wilcox_score,log2fc,pvals,padj,pts_group,pts_ref,group,annotation,log2fc_edger,stat_edger,pval_edger,padj_edger,sc_signicant,psc_signicant,consensus_significant,MeanExpr_batch1,MeanExpr_batch2
0,RPS4Y1,10.417766,32.312714,2.057288e-25,9.523701e-22,0.812865,0.000000,disease,B_cells,9.811563,241.036251,2.109039e-08,4.348838e-06,Yes,Yes,Yes,0.000000,1.846016
1,CD83,10.117800,4.033925,4.606540e-24,1.421655e-20,0.865497,0.166667,disease,B_cells,3.909144,253.064918,5.244110e-09,1.544765e-06,Yes,Yes,Yes,0.549657,2.565062
2,JUND,9.986894,2.114214,1.739467e-23,4.601388e-20,0.959064,0.916667,disease,B_cells,2.107018,381.290324,5.662320e-10,2.746877e-07,Yes,Yes,Yes,2.782248,4.198936
3,FOS,9.884114,4.776987,4.878770e-23,1.129252e-19,0.883041,0.305556,disease,B_cells,4.660104,851.268223,7.002705e-12,1.443958e-08,Yes,Yes,Yes,0.981424,3.844560
4,HSP90AA1,9.636548,3.000897,5.604088e-22,1.153010e-18,0.929825,0.597222,disease,B_cells,3.051359,417.310442,3.469072e-10,2.229698e-07,Yes,Yes,Yes,1.463146,3.316828
5,CREM,9.293894,5.343356,1.487451e-20,2.754314e-17,0.748538,0.055556,disease,B_cells,5.283806,370.997592,5.994640e-10,2.746877e-07,Yes,Yes,Yes,0.192004,2.261108
6,DUSP2,8.601924,7.102722,7.839049e-18,1.036826e-14,0.660819,0.027778,disease,B_cells,6.899147,571.830847,1.453186e-11,1.997646e-08,Yes,Yes,Yes,0.074408,2.452486
7,RGS1,8.586860,7.340136,8.937884e-18,1.103352e-14,0.654971,0.027778,disease,B_cells,7.330024,696.751552,4.485111e-12,1.443958e-08,Yes,Yes,Yes,0.076572,2.631500
8,YPEL5,8.577455,2.700320,9.699565e-18,1.122543e-14,0.853801,0.347222,disease,B_cells,2.794429,236.244363,7.485382e-09,2.028616e-06,Yes,Yes,Yes,0.964321,2.446569
9,EIF1,8.334270,0.915142,7.797797e-17,8.493636e-14,0.988304,1.000000,disease,B_cells,0.980164,79.229647,2.120776e-06,1.016986e-04,Yes,Yes,Yes,3.039560,3.651153


In [4]:
adata

AnnData object with n_obs × n_vars = 2801 × 18517
    obs: 'batch', 'condition', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'n_counts', 'doublet_class', 'doublet_score', 'leiden', 'cell_type', 'autoAnnot', 'celltypist_conf_score', 'annotation', 'annotation_recluster'
    var: 'mean', 'std', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'annotation_colors', 'annotation_recluster_colors', 'batch_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap', 'rank_genes_condition', 'rank_genes_deseq2', 'rank_genes_consensus'
    obsm: 'X_CCA', 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'logcounts', 'scaled'
    obsp: 'connectivities', 'distances'

As we can appreciate, the results of the DEA will be saved in the uns attributed

In [5]:
!pip list

Package                       Version        Editable project location
----------------------------- -------------- ---------------------------------------------------
absl-py                       2.2.2
accessible-pygments           0.0.5
adjustText                    1.3.0
aiobotocore                   2.21.1
aiohappyeyeballs              2.6.1
aiohttp                       3.11.16
aioitertools                  0.12.0
aiosignal                     1.3.2
alabaster                     1.0.0
anndata                       0.11.4
annotated-types               0.7.0
annoy                         1.17.3
anyio                         4.7.0
appnope                       0.1.2
archspec                      0.2.3
argon2-cffi                   21.3.0
argon2-cffi-bindings          21.2.0
array_api_compat              1.11.2
arrow                         1.3.0
asciitree                     0.3.3
asttokens                     3.0.0
async-lru                     2.0.4
async-timeout                 5